In [1]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM CMCAS10", con=connection2)

connection2.close()




In [2]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



#connection3.execute('CREATE TEMPORARY TABLE tmp_cmcas10 ()')
base2.to_sql(name='tmp_cmcas10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cmcas10 FALSO CON LO OBTENIDO DEL ORACLE

query="""

ALTER TABLE tmp_cmcas10

ALTER COLUMN oricenasicod TYPE character(1),
ALTER COLUMN cenasicod TYPE character(3),
ALTER COLUMN cenasides TYPE character(100),
ALTER COLUMN cenasidescor TYPE character(20),
ALTER COLUMN cenasidir TYPE character(100),
ALTER COLUMN tipcentasiscod TYPE character(2),
ALTER COLUMN nivcentasiscod TYPE character(1),
ALTER COLUMN clacentasiscod TYPE character(1),
ALTER COLUMN redasiscod TYPE character(2),
ALTER COLUMN cenasialmlogsapcod TYPE character(4),
ALTER COLUMN cenasicensapcod TYPE character(4),
ALTER COLUMN cenasiuniopesapcod TYPE character(4),
ALTER COLUMN estregcod TYPE character(1),
ALTER COLUMN cenasisisflg TYPE character(1),
ALTER COLUMN cenasiimgprvflg TYPE character(1),
ALTER COLUMN cenasilabeqpflg TYPE character(1),
ALTER COLUMN cenasirenaescod TYPE character(8),
ALTER COLUMN cenasiateredflg TYPE character(1),
ALTER COLUMN cenasirenaesunigescod TYPE character(8),
ALTER COLUMN cenasidecurgflg TYPE character(1),
ALTER COLUMN cenasiusucrecod TYPE character(10),
ALTER COLUMN cenasicrefec TYPE date,
ALTER COLUMN cenasiusumodcod TYPE character(10),
ALTER COLUMN cenasimodfec TYPE date,
ALTER COLUMN cenasitelef1 TYPE character(15),
ALTER COLUMN cenasilongitud TYPE character(100),
ALTER COLUMN cenasilatitud TYPE character(100),
ALTER COLUMN cenasihemoflg TYPE character(1),
ALTER COLUMN cenasicatcenasicod TYPE character(2),
ALTER COLUMN cenasidepcod TYPE character(2);



UPDATE cmcas10 
SET 
oricenasicod=t.oricenasicod
,cenasicod=t.cenasicod
,cenasides=t.cenasides
,cenasidescor=t.cenasidescor
,cenasidir=t.cenasidir
,tipcentasiscod=t.tipcentasiscod
,nivcentasiscod=t.nivcentasiscod
,clacentasiscod=t.clacentasiscod
,redasiscod=t.redasiscod
,cenasialmlogsapcod=t.cenasialmlogsapcod
,cenasicensapcod=t.cenasicensapcod
,cenasiuniopesapcod=t.cenasiuniopesapcod
,estregcod=t.estregcod
,cenasisisflg=t.cenasisisflg
,cenasiimgprvflg=t.cenasiimgprvflg
,cenasilabeqpflg=t.cenasilabeqpflg
,cenasirenaescod=t.cenasirenaescod
,cenasiateredflg=t.cenasiateredflg
,cenasirenaesunigescod=t.cenasirenaesunigescod
,cenasidecurgflg=t.cenasidecurgflg
,cenasiusucrecod=t.cenasiusucrecod
,cenasicrefec=t.cenasicrefec
,cenasiusumodcod=t.cenasiusumodcod
,cenasimodfec=t.cenasimodfec
,cenasitelef1=t.cenasitelef1
,cenasilongitud=t.cenasilongitud
,cenasilatitud=t.cenasilatitud
,cenasihemoflg=t.cenasihemoflg
,cenasicatcenasicod=t.cenasicatcenasicod
,cenasidepcod=t.cenasidepcod     

FROM tmp_cmcas10 t 
WHERE cmcas10.oricenasicod = t.oricenasicod and cmcas10.cenasicod = t.cenasicod and cmcas10.cenasicod IS NOT NULL and t.cenasicod IS NOT NULL ;

INSERT INTO cmcas10 
(oricenasicod, cenasicod, cenasides, cenasidescor, cenasidir,
tipcentasiscod, nivcentasiscod, clacentasiscod, redasiscod,
cenasialmlogsapcod, cenasicensapcod, cenasiuniopesapcod,
estregcod, cenasisisflg, cenasiimgprvflg, cenasilabeqpflg,
cenasirenaescod, cenasiateredflg, cenasirenaesunigescod,
cenasidecurgflg, cenasiusucrecod, cenasicrefec, cenasiusumodcod,
cenasimodfec, cenasitelef1, cenasilongitud, cenasilatitud,
cenasihemoflg, cenasicatcenasicod, cenasidepcod) 
SELECT 
oricenasicod, cenasicod, cenasides, cenasidescor, cenasidir,
tipcentasiscod, nivcentasiscod, clacentasiscod, redasiscod,
cenasialmlogsapcod, cenasicensapcod, cenasiuniopesapcod,
estregcod, cenasisisflg, cenasiimgprvflg, cenasilabeqpflg,
cenasirenaescod, cenasiateredflg, cenasirenaesunigescod,
cenasidecurgflg, cenasiusucrecod, cenasicrefec, cenasiusumodcod,
cenasimodfec, cenasitelef1, cenasilongitud, cenasilatitud,
cenasihemoflg, cenasicatcenasicod, cenasidepcod

FROM tmp_cmcas10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM cmcas10 
    WHERE cmcas10.oricenasicod = tmp_cmcas10.oricenasicod and cmcas10.cenasicod = tmp_cmcas10.cenasicod and cmcas10.cenasicod IS NOT NULL and tmp_cmcas10.cenasicod IS NOT NULL
) ;


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_cmcas10;
DELETE FROM cmcas10 WHERE cenasicod ='';
"""


c2= text(query2)
connection3.execute(c2)
base2 = pd.read_sql_query(f"SELECT * FROM cmcas10", con=connection3)

connection3.close()


In [3]:
base2.columns

Index(['oricenasicod', 'cenasicod', 'cenasides', 'cenasidescor', 'cenasidir',
       'tipcentasiscod', 'nivcentasiscod', 'clacentasiscod', 'redasiscod',
       'cenasialmlogsapcod', 'cenasicensapcod', 'cenasiuniopesapcod',
       'estregcod', 'cenasisisflg', 'cenasiimgprvflg', 'cenasilabeqpflg',
       'cenasirenaescod', 'cenasiateredflg', 'cenasirenaesunigescod',
       'cenasidecurgflg', 'cenasiusucrecod', 'cenasicrefec', 'cenasiusumodcod',
       'cenasimodfec', 'cenasitelef1', 'cenasilongitud', 'cenasilatitud',
       'cenasihemoflg', 'cenasicatcenasicod', 'cenasidepcod'],
      dtype='object')

In [4]:
#################################################################################################################################################################################################################################################################################


#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_cmcas10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query="""

UPDATE dim_cas 
SET 

cod_red = t.redasiscod,
ori_cas = t.oricenasicod,
cod_cas = t.cenasicod,
tip_cas = t.tipcentasiscod,
niv_cas = t.nivcentasiscod,
cla_cas = t.clacentasiscod

FROM tmp_cmcas10 t 
WHERE dim_cas.cod_cas = t.cenasicod AND dim_cas.ori_cas=t.oricenasicod AND dim_cas.cod_cas  IS NOT NULL;


INSERT INTO dim_cas (cod_cas, ori_cas, cod_red,des_cas,dco_cas,dir_cas,tip_cas,niv_cas,cla_cas) 
SELECT cenasicod, oricenasicod, redasiscod, cenasides, cenasidescor, cenasidir,tipcentasiscod, nivcentasiscod, clacentasiscod      

FROM tmp_cmcas10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_cas 
    WHERE dim_cas.cod_cas = tmp_cmcas10.cenasicod and dim_cas.ori_cas = tmp_cmcas10.oricenasicod
);

DROP TABLE tmp_cmcas10;

"""

c1= text(query)
connection4.execute(c1)
connection4.close()